# LeetCode #15 — 3Sum: Sắp xếp + Two Pointers + bỏ trùng lặp

**Mức độ:** Medium · **Chủ đề:** Array, Two Pointers, Sorting

Skeleton trên LeetCode:

```python
class Solution:
    def threeSum(self, nums: List[int]) -> List[List[int]]:
```

3Sum là bài "nâng cấp" của Two Sum, và là một trong những bài phỏng vấn nổi tiếng nhất. Cái khó **không nằm ở việc tìm** bộ ba, mà ở việc **không trả về bộ trùng lặp** một cách hiệu quả.

Mục tiêu của notebook:

1. Biến bài 3 số thành **nhiều bài 2 số** bằng cách cố định một phần tử.
2. Dùng **sắp xếp** để vừa áp dụng Two Pointers, vừa bỏ trùng dễ dàng.
3. Hiểu chính xác **chỗ nào cần bỏ trùng** và vì sao.

# 1. Bài toán là gì?

Cho mảng số nguyên `nums`. Trả về **mọi bộ ba** `[nums[i], nums[j], nums[k]]` sao cho:

- `i`, `j`, `k` đôi một khác nhau,
- `nums[i] + nums[j] + nums[k] == 0`.

Kết quả **không được chứa bộ ba trùng nhau** (hai bộ ba trùng nếu có cùng các giá trị, bất kể thứ tự). Thứ tự các bộ ba và thứ tự trong mỗi bộ ba không quan trọng.

**Constraints (theo đề):**

- `3 <= nums.length <= 3000`
- `-10^5 <= nums[i] <= 10^5`

---

## Ví dụ

```text
nums = [-1, 0, 1, 2, -1, -4]  ->  [[-1, -1, 2], [-1, 0, 1]]
nums = [0, 1, 1]              ->  []
nums = [0, 0, 0]              ->  [[0, 0, 0]]
```

Ở ví dụ 1, `[-1, 0, 1]` có thể tạo từ chỉ số `(0, 1, 2)` hoặc `(4, 1, 2)` — nhưng chỉ được xuất hiện **một lần**.

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Sắp xếp (sorting)
2. Hai con trỏ (two pointers)
3. Bảng băm (hash table) — `dict` và `set`

Cuối mục có phần **2.4 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Sắp xếp (sorting)

### Sắp xếp trong Python

- `sorted(a)` trả về **list mới**; `a.sort()` sắp xếp **tại chỗ** (trả về `None`).
- `key=` chọn tiêu chí: `sorted(words, key=len)`, `sorted(pairs, key=lambda p: p[1])`.
- `reverse=True` để giảm dần; sắp xếp theo nhiều tiêu chí bằng `key` trả về tuple.
- Thuật toán bên trong là **Timsort**: `O(n log n)`, **ổn định** (stable — hai phần tử bằng nhau giữ nguyên thứ tự ban đầu).

In [1]:
a_sort = [5, 2, 9, 1, 5, 6]
print("sorted:", sorted(a_sort), "| gốc không đổi:", a_sort)
a_sort.sort(reverse=True)
print("sort giảm dần tại chỗ:", a_sort)

nguoi = [("An", 25), ("Bình", 20), ("Chi", 25), ("Dũng", 20)]
print("theo tuổi (ổn định — An trước Chi):", sorted(nguoi, key=lambda p: p[1]))
print("tuổi giảm, rồi tên tăng:", sorted(nguoi, key=lambda p: (-p[1], p[0])))
print("chuỗi sắp theo ký tự:", "".join(sorted("banana")))

sorted: [1, 2, 5, 5, 6, 9] | gốc không đổi: [5, 2, 9, 1, 5, 6]
sort giảm dần tại chỗ: [9, 6, 5, 5, 2, 1]
theo tuổi (ổn định — An trước Chi): [('Bình', 20), ('Dũng', 20), ('An', 25), ('Chi', 25)]
tuổi giảm, rồi tên tăng: [('An', 25), ('Chi', 25), ('Bình', 20), ('Dũng', 20)]
chuỗi sắp theo ký tự: aaabnn


### Vì sao sắp xếp hay là bước đầu tiên?

Sau khi sắp xếp: phần tử **bằng nhau đứng cạnh nhau** (dễ bỏ trùng), có thể dùng **hai con trỏ** / **tìm nhị phân**, các khoảng chồng nhau **liền kề** nhau. Đổi lại tốn `O(n log n)`.

### Sắp xếp chèn (insertion sort)

Giống xếp bài trên tay: lấy lần lượt từng phần tử, **chèn** vào đúng vị trí trong phần đã sắp xếp bên trái. `O(n²)` xấu nhất, nhưng `O(n)` khi dữ liệu gần như đã sắp xếp.

In [2]:
def sap_xep_chen(a):
    a = a[:]
    for i in range(1, len(a)):
        x, j = a[i], i - 1
        while j >= 0 and a[j] > x:
            a[j + 1] = a[j]            # dịch phần tử lớn hơn sang phải
            j -= 1
        a[j + 1] = x
        print(f"   sau khi chèn {x}: {a[:i + 1]} | {a[i + 1:]}")
    return a

print("→", sap_xep_chen([5, 2, 4, 6, 1, 3]))

   sau khi chèn 2: [2, 5] | [4, 6, 1, 3]
   sau khi chèn 4: [2, 4, 5] | [6, 1, 3]
   sau khi chèn 6: [2, 4, 5, 6] | [1, 3]
   sau khi chèn 1: [1, 2, 4, 5, 6] | [3]
   sau khi chèn 3: [1, 2, 3, 4, 5, 6] | []
→ [1, 2, 3, 4, 5, 6]


### Sắp xếp trộn (merge sort) và phép trộn hai dãy đã sắp xếp

**Chia để trị**: chia đôi, sắp xếp mỗi nửa (đệ quy), rồi **trộn** hai nửa đã sắp xếp. Trộn: hai con trỏ ở đầu hai dãy, mỗi bước lấy phần tử nhỏ hơn. `O(n log n)` mọi trường hợp, ổn định (nhờ lấy bên trái khi bằng nhau).

```text
[5,2,4,6,1,3] → [5,2,4] [6,1,3] → [5] [2,4] [6] [1,3] → … → [2,4,5] [1,3,6] → [1,2,3,4,5,6]
```

In [3]:
def tron(x, y):
    i = j = 0
    out = []
    while i < len(x) and j < len(y):
        if x[i] <= y[j]:               # <= giữ ổn định
            out.append(x[i]); i += 1
        else:
            out.append(y[j]); j += 1
    out.extend(x[i:]); out.extend(y[j:])
    return out

def sap_xep_tron(a, sau=0):
    if len(a) <= 1:
        return a
    mid = len(a) // 2
    trai, phai = sap_xep_tron(a[:mid], sau + 1), sap_xep_tron(a[mid:], sau + 1)
    kq = tron(trai, phai)
    print("    " * sau + f"trộn {trai} + {phai} = {kq}")
    return kq

print("→", sap_xep_tron([5, 2, 4, 6, 1, 3]))

        trộn [2] + [4] = [2, 4]
    trộn [5] + [2, 4] = [2, 4, 5]
        trộn [1] + [3] = [1, 3]
    trộn [6] + [1, 3] = [1, 3, 6]
trộn [2, 4, 5] + [1, 3, 6] = [1, 2, 3, 4, 5, 6]
→ [1, 2, 3, 4, 5, 6]


### Bảng so sánh

| Thuật toán | Tốt nhất | Xấu nhất | Bộ nhớ phụ | Ổn định |
|---|---|---|---|---|
| Insertion sort | `O(n)` | `O(n²)` | `O(1)` | có |
| Merge sort | `O(n log n)` | `O(n log n)` | `O(n)` (mảng) | có |
| Quick sort | `O(n log n)` | `O(n²)` | `O(log n)` | không |
| Timsort (`sorted`) | `O(n)` | `O(n log n)` | `O(n)` | có |

Lỗi hay gặp: `a = a.sort()` (gán `None`); quên rằng `sorted` tạo list mới (tốn bộ nhớ); `key` trả về kiểu không so sánh được.

## 2.2 Hai con trỏ (two pointers)

### Ý tưởng

Dùng **hai chỉ số** di chuyển trên mảng/chuỗi theo quy tắc, để thay vòng lặp lồng `O(n²)` bằng một lượt `O(n)`. Hai kiểu chính:

**1. Hai đầu tiến vào nhau** (thường trên mảng đã sắp xếp):

```text
[1, 2, 4, 7, 11, 15], tìm hai số có tổng 15
 L                 R     1 + 15 = 16 > 15 → R lùi (cần nhỏ hơn)
 L             R         1 + 11 = 12 < 15 → L tiến (cần lớn hơn)
    L          R         2 + 11 = 13 < 15 → L tiến
       L       R         4 + 11 = 15 ✓
```

Vì sao được phép bỏ? Khi `a[L] + a[R] > target`, mọi cặp `(L', R)` với `L' > L` còn lớn hơn → `R` không thể là đáp án, bỏ `R`.

**2. Cùng chiều — con trỏ đọc và con trỏ ghi** (sửa mảng tại chỗ):

```text
xóa mọi số 0, giữ thứ tự: [0, 1, 0, 3, 12]
read đi qua từng phần tử; gặp số khác 0 thì chép vào vị trí write, write += 1
→ [1, 3, 12, ...]
```

In [4]:
def tong_hai_so_da_sap_xep(a, target):
    L, R = 0, len(a) - 1
    while L < R:
        s = a[L] + a[R]
        print(f"   L={L} R={R}: {a[L]} + {a[R]} = {s}")
        if s == target:
            return L, R
        if s < target:
            L += 1
        else:
            R -= 1
    return None

print("→", tong_hai_so_da_sap_xep([1, 2, 4, 7, 11, 15], 15))

def bo_so_khong(a):
    write = 0
    for read in range(len(a)):
        if a[read] != 0:
            a[write] = a[read]
            write += 1
    return a[:write]

print("bỏ số 0:", bo_so_khong([0, 1, 0, 3, 12]))

def la_palindrome(s):
    i, j = 0, len(s) - 1
    while i < j:
        if s[i] != s[j]:
            return False
        i += 1; j -= 1
    return True
print("'racecar' palindrome?", la_palindrome("racecar"), "| 'abca'?", la_palindrome("abca"))

   L=0 R=5: 1 + 15 = 16
   L=0 R=4: 1 + 11 = 12
   L=1 R=4: 2 + 11 = 13
   L=2 R=4: 4 + 11 = 15
→ (2, 4)
bỏ số 0: [1, 3, 12]
'racecar' palindrome? True | 'abca'? False


### Đảo ngược một đoạn tại chỗ

Hai con trỏ ở hai đầu, đổi chỗ rồi tiến vào trong — dùng rất nhiều (đảo mảng, xoay mảng, đảo từ).

In [5]:
def dao_doan(a, i, j):
    while i < j:
        a[i], a[j] = a[j], a[i]
        i += 1; j -= 1

arr_tp = [1, 2, 3, 4, 5, 6]
dao_doan(arr_tp, 1, 4)
print("đảo đoạn [1..4]:", arr_tp)

đảo đoạn [1..4]: [1, 5, 4, 3, 2, 6]


### Lỗi hay gặp

- Quên **sắp xếp** trước khi dùng hai đầu cho bài tổng.
- Điều kiện `L <= R` thay vì `L < R` → dùng một phần tử hai lần.
- Không chứng minh được vì sao bỏ một đầu là an toàn → dễ bỏ mất đáp án.
- Con trỏ ghi vượt con trỏ đọc (không thể xảy ra nếu `write` chỉ tăng khi ghi).

## 2.3 Bảng băm (hash table) — `dict` và `set`

### Bảng băm là gì?

Bảng băm lưu các cặp **khóa → giá trị** sao cho **tra cứu, thêm, xóa theo khóa** chỉ mất trung bình `O(1)`.

Ý tưởng: một **hàm băm** biến khóa thành một con số; con số đó chỉ ra **ngăn** (bucket) trong một mảng nơi cất giá trị. Thay vì dò lần lượt cả danh sách (`O(n)`), ta tính hàm băm rồi nhảy thẳng tới ngăn.

```text
khóa "cam"  --hash--> 7  ──►  ngăn[7] = ("cam", 3)
khóa "táo"  --hash--> 2  ──►  ngăn[2] = ("táo", 5)
khóa "lê"   --hash--> 7  ──►  ngăn[7] đã có "cam" → VA CHẠM, Python tự xử lý (dò ngăn khác)
```

Python có sẵn hai kiểu dựa trên bảng băm:

| Kiểu | Lưu gì | Ví dụ |
|---|---|---|
| `dict` | khóa → giá trị | `{"cam": 3, "táo": 5}` |
| `set` | chỉ khóa (tập hợp, không trùng) | `{1, 4, 9}` |

**Khóa phải "hashable"** (bất biến): `int`, `str`, `tuple`, `frozenset` dùng được; `list`, `dict`, `set` thì **không**.

In [6]:
# --- dict: khóa -> giá trị ---
gia = {"cam": 3, "táo": 5}
gia["lê"] = 4                     # thêm
gia["cam"] = 6                    # cập nhật
print("gia['táo'] =", gia["táo"])
print("'xoài' có trong dict?", "xoài" in gia)        # kiểm tra khóa: O(1)
print("get có giá trị mặc định:", gia.get("xoài", 0))
del gia["lê"]                     # xóa
for k, v in gia.items():          # duyệt cặp khóa-giá trị
    print(" ", k, "->", v)

# --- set: tập hợp không trùng ---
s = {3, 1, 3, 2}
s.add(5); s.discard(1)
print("set:", s, "| 2 in s:", 2 in s)
print("giao:", {1, 2, 3} & {2, 3, 4}, "| hợp:", {1, 2} | {5}, "| hiệu:", {1, 2, 3} - {2})

gia['táo'] = 5
'xoài' có trong dict? False
get có giá trị mặc định: 0
  cam -> 6
  táo -> 5
set: {2, 3, 5} | 2 in s: True
giao: {2, 3} | hợp: {1, 2, 5} | hiệu: {1, 3}


### Vì sao nhanh? So sánh `list` và `set` khi kiểm tra phần tử

`x in list` phải so sánh lần lượt từng phần tử (`O(n)`); `x in set` chỉ tính hàm băm (`O(1)` trung bình).

In [7]:
import time as _time_hash

_data = list(range(200_000))
_as_set = set(_data)
_t0 = _time_hash.perf_counter()
for _x in range(199_000, 199_100):
    _ = _x in _data
_t1 = _time_hash.perf_counter()
for _x in range(199_000, 199_100):
    _ = _x in _as_set
_t2 = _time_hash.perf_counter()
print(f"100 lần 'in' trên list: {(_t1 - _t0) * 1000:.2f} ms | trên set: {(_t2 - _t1) * 1000:.3f} ms")

100 lần 'in' trên list: 121.14 ms | trên set: 0.096 ms


### Bốn mẫu dùng bảng băm hay gặp nhất

1. **Đếm tần suất** — `Counter` hoặc `d[x] = d.get(x, 0) + 1`.
2. **Nhóm theo khóa** — `defaultdict(list)`, khóa là "đặc trưng chung" của nhóm.
3. **Ghi nhớ thứ đã thấy** — lưu giá trị → vị trí để trả lời "đã gặp phần bù chưa?" trong `O(1)` (kiểu Two Sum).
4. **Đánh dấu đã thăm** — `visited = set()` khi duyệt đồ thị / lưới.

In [8]:
from collections import Counter, defaultdict

# 1) Đếm
print("Counter:", Counter("banana"))

# 2) Nhóm theo khóa: nhóm các từ theo chữ cái đầu
nhom = defaultdict(list)
for w in ["an", "bò", "ba", "ăn", "ai"]:
    nhom[w[0]].append(w)
print("Nhóm:", dict(nhom))

# 3) Ghi nhớ đã thấy: tìm hai chỉ số có tổng = target trong một lượt
def hai_so_tong(nums, target):
    vi_tri = {}                       # giá trị -> chỉ số
    for i, x in enumerate(nums):
        if target - x in vi_tri:      # phần bù đã gặp trước đó?
            return vi_tri[target - x], i
        vi_tri[x] = i
    return None
print("hai_so_tong([2, 7, 11, 15], 9) =", hai_so_tong([2, 7, 11, 15], 9))

# 4) Đã thăm
da_tham = set()
for v in [3, 1, 3, 2, 1]:
    if v in da_tham:
        print("gặp lại", v)
    da_tham.add(v)

Counter: Counter({'a': 3, 'n': 2, 'b': 1})
Nhóm: {'a': ['an', 'ai'], 'b': ['bò', 'ba'], 'ă': ['ăn']}
hai_so_tong([2, 7, 11, 15], 9) = (0, 1)
gặp lại 3
gặp lại 1


### Độ phức tạp và lưu ý

| Thao tác | `dict` / `set` (trung bình) | `list` |
|---|---|---|
| kiểm tra `x in ...` | `O(1)` | `O(n)` |
| thêm / xóa theo khóa | `O(1)` | xóa giữa: `O(n)` |
| duyệt toàn bộ | `O(n)` | `O(n)` |

Lỗi hay gặp:

- Dùng `list` làm khóa → `TypeError: unhashable type`. Đổi sang `tuple(...)`.
- Thêm/xóa khóa **trong lúc** đang duyệt chính dict đó → `RuntimeError`.
- Truy cập `d[k]` khi `k` chưa có → `KeyError`; dùng `d.get(k, mặc_định)` hoặc `defaultdict`.

In [9]:
try:
    {[1, 2]: "x"}
except TypeError as e:
    print("Khóa là list:", e)
print("Khóa là tuple thì được:", {(1, 2): "x"})

Khóa là list: unhashable type: 'list'
Khóa là tuple thì được: {(1, 2): 'x'}


## 2.4 Áp dụng vào bài này

**Sắp xếp** trước để: (1) các số bằng nhau đứng cạnh nhau, bỏ trùng dễ dàng; (2) cố định số thứ nhất rồi tìm hai số còn lại bằng **hai con trỏ ở hai đầu** — biến `O(n³)` thành `O(n²)`. Cách dùng **bảng băm** (tìm phần bù) cũng được nhưng khó bỏ trùng hơn.

# 3. Nền tảng

## 3.1 Sắp xếp và tác dụng với trùng lặp

`sorted(nums)` hoặc `nums.sort()` xếp tăng dần. Sau khi sắp xếp, **các giá trị bằng nhau nằm cạnh nhau** — muốn bỏ qua giá trị trùng chỉ cần so với phần tử đứng trước.

In [10]:
nums = [-1, 0, 1, 2, -1, -4]
nums_sorted = sorted(nums)
print(nums_sorted)

for i, x in enumerate(nums_sorted):
    is_repeat = i > 0 and x == nums_sorted[i - 1]
    print(f"i={i}, x={x:>2} {'(trùng với phần tử trước -> bỏ qua)' if is_repeat else ''}")

[-4, -1, -1, 0, 1, 2]
i=0, x=-4 
i=1, x=-1 
i=2, x=-1 (trùng với phần tử trước -> bỏ qua)
i=3, x= 0 
i=4, x= 1 
i=5, x= 2 


## 3.2 Chuẩn hóa một bộ ba để so sánh

Hai bộ ba `[-1, 0, 1]` và `[0, 1, -1]` là **một**. Nếu sắp xếp mỗi bộ ba và đổi sang `tuple`, ta có thể bỏ vào `set` để loại trùng.

`tuple` (khác `list`) có thể làm phần tử của `set`.

In [11]:
a = tuple(sorted([0, 1, -1]))
b = tuple(sorted([-1, 0, 1]))
print(a, b, a == b)

seen = {a}
print(b in seen)

(-1, 0, 1) (-1, 0, 1) True
True


# 4. Cách dễ nghĩ nhất: thử mọi bộ ba

Ba vòng lặp lồng nhau, bỏ trùng bằng `set` các tuple đã sắp xếp.

In [12]:
from typing import List

def three_sum_bruteforce(nums: List[int]) -> List[List[int]]:
    n = len(nums)
    found = set()
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                if nums[i] + nums[j] + nums[k] == 0:
                    found.add(tuple(sorted((nums[i], nums[j], nums[k]))))
    return [list(t) for t in sorted(found)]

print(three_sum_bruteforce([-1, 0, 1, 2, -1, -4]))
print(three_sum_bruteforce([0, 1, 1]))
print(three_sum_bruteforce([0, 0, 0]))

[[-1, -1, 2], [-1, 0, 1]]
[]
[[0, 0, 0]]


**Độ phức tạp:** `O(n³)`. Với `n = 3000` là khoảng `4.5 * 10^9` bộ ba — quá chậm.

# 5. Cải tiến lần 1: cố định một số, còn lại là Two Sum

Nếu cố định `a = nums[i]`, ta cần tìm hai số còn lại có tổng `-a`. Đó chính là **Two Sum** với `target = -a`!

Dùng Hash Set như bài #1 cho phần Two Sum → `O(n)` mỗi lần, tổng `O(n²)`.

Nhưng bỏ trùng vẫn phải dùng `set` các tuple — được, nhưng tốn bộ nhớ và hơi "mù quáng".

In [13]:
def three_sum_hashset(nums: List[int]) -> List[List[int]]:
    n = len(nums)
    found = set()
    for i in range(n):
        target = -nums[i]
        seen = set()
        for j in range(i + 1, n):
            needed = target - nums[j]
            if needed in seen:
                found.add(tuple(sorted((nums[i], nums[j], needed))))
            seen.add(nums[j])
    return [list(t) for t in sorted(found)]

print(three_sum_hashset([-1, 0, 1, 2, -1, -4]))

[[-1, -1, 2], [-1, 0, 1]]


# 6. Ý tưởng chính: sắp xếp + Two Pointers

Sau khi **sắp xếp**, Two Sum trên đoạn `nums[i+1 .. n-1]` có thể làm bằng **hai con trỏ** (giống bài #167):

```text
left = i + 1, right = n - 1
total = nums[i] + nums[left] + nums[right]
- total < 0  -> cần lớn hơn -> left += 1
- total > 0  -> cần nhỏ hơn -> right -= 1
- total == 0 -> ghi nhận, rồi dời CẢ HAI con trỏ
```

Vì sao dời đúng? Mảng tăng dần: tăng `left` làm tổng **không giảm**, giảm `right` làm tổng **không tăng**.

## Ba chỗ bỏ trùng

Vì mảng đã sắp xếp, các giá trị bằng nhau đứng cạnh nhau:

1. **Số cố định `nums[i]`:** nếu `nums[i] == nums[i - 1]`, mọi bộ ba bắt đầu bằng giá trị này **đã được tìm** ở lần trước → `continue`.
2. **Sau khi tìm được một bộ ba:** dời `left` qua mọi phần tử bằng `nums[left]`, (và `right` qua mọi phần tử bằng `nums[right]`), để không ghi lại cùng bộ ba.
3. Không cần bỏ trùng khi `total != 0` — chỉ dời một bước là đủ (có trùng cũng chỉ tốn thêm vài bước, không sinh kết quả trùng).

## Một tối ưu nhỏ

Nếu `nums[i] > 0`: ba số đều dương (mảng tăng dần) → tổng > 0 → **dừng hẳn**.

In [14]:
def three_sum(nums: List[int]) -> List[List[int]]:
    nums = sorted(nums)
    n = len(nums)
    result = []

    for i in range(n - 2):
        if nums[i] > 0:
            break                               # không thể có tổng 0 nữa
        if i > 0 and nums[i] == nums[i - 1]:
            continue                            # bỏ trùng số cố định

        left, right = i + 1, n - 1
        while left < right:
            total = nums[i] + nums[left] + nums[right]
            if total < 0:
                left += 1
            elif total > 0:
                right -= 1
            else:
                result.append([nums[i], nums[left], nums[right]])
                left += 1
                right -= 1
                while left < right and nums[left] == nums[left - 1]:
                    left += 1                   # bỏ trùng phần tử thứ hai
                while left < right and nums[right] == nums[right + 1]:
                    right -= 1                  # bỏ trùng phần tử thứ ba

    return result

print(three_sum([-1, 0, 1, 2, -1, -4]))
print(three_sum([0, 1, 1]))
print(three_sum([0, 0, 0]))
print(three_sum([-2, 0, 0, 2, 2]))

[[-1, -1, 2], [-1, 0, 1]]
[]
[[0, 0, 0]]
[[-2, 0, 2]]


# 7. Mô phỏng từng bước

`nums = [-1, 0, 1, 2, -1, -4]` → sắp xếp: `[-4, -1, -1, 0, 1, 2]` (chỉ số 0..5).

| i (giá trị) | left | right | total | Hành động |
|---|---|---|---|---|
| 0 (-4) | 1 (-1) | 5 (2) | -3 | < 0 → left++ |
| 0 (-4) | 2 (-1) | 5 (2) | -3 | left++ |
| 0 (-4) | 3 (0) | 5 (2) | -2 | left++ |
| 0 (-4) | 4 (1) | 5 (2) | -1 | left++ → left = right, dừng |
| 1 (-1) | 2 (-1) | 5 (2) | 0 | **ghi [-1,-1,2]**, left=3, right=4 |
| 1 (-1) | 3 (0) | 4 (1) | 0 | **ghi [-1,0,1]**, left=4, right=3, dừng |
| 2 (-1) | | | | trùng với i=1 → **bỏ qua** |
| 3 (0) | 4 (1) | 5 (2) | 3 | > 0 → right--, dừng |

Ở `i = 2`, nếu không bỏ qua, ta sẽ ghi lại `[-1, 0, 1]` lần nữa.

In [15]:
def trace_three_sum(nums: List[int]) -> List[List[int]]:
    nums = sorted(nums)
    print("Sau sắp xếp:", nums)
    n = len(nums)
    result = []
    for i in range(n - 2):
        if nums[i] > 0:
            print(f"i={i}: nums[i]={nums[i]} > 0 -> dừng hẳn")
            break
        if i > 0 and nums[i] == nums[i - 1]:
            print(f"i={i}: nums[i]={nums[i]} trùng phần tử trước -> bỏ qua")
            continue
        left, right = i + 1, n - 1
        while left < right:
            total = nums[i] + nums[left] + nums[right]
            line = f"i={i}({nums[i]}) left={left}({nums[left]}) right={right}({nums[right]}) total={total:>2}"
            if total < 0:
                print(line, "-> left += 1")
                left += 1
            elif total > 0:
                print(line, "-> right -= 1")
                right -= 1
            else:
                triple = [nums[i], nums[left], nums[right]]
                result.append(triple)
                print(line, f"-> GHI {triple}")
                left += 1
                right -= 1
                while left < right and nums[left] == nums[left - 1]:
                    print(f"    bỏ trùng left tại {left} ({nums[left]})")
                    left += 1
                while left < right and nums[right] == nums[right + 1]:
                    print(f"    bỏ trùng right tại {right} ({nums[right]})")
                    right -= 1
    return result

print("Kết quả:", trace_three_sum([-1, 0, 1, 2, -1, -4]))

Sau sắp xếp: [-4, -1, -1, 0, 1, 2]
i=0(-4) left=1(-1) right=5(2) total=-3 -> left += 1
i=0(-4) left=2(-1) right=5(2) total=-3 -> left += 1
i=0(-4) left=3(0) right=5(2) total=-2 -> left += 1
i=0(-4) left=4(1) right=5(2) total=-1 -> left += 1
i=1(-1) left=2(-1) right=5(2) total= 0 -> GHI [-1, -1, 2]
i=1(-1) left=3(0) right=4(1) total= 0 -> GHI [-1, 0, 1]
i=2: nums[i]=-1 trùng phần tử trước -> bỏ qua
i=3(0) left=4(1) right=5(2) total= 3 -> right -= 1
Kết quả: [[-1, -1, 2], [-1, 0, 1]]


Thử một ví dụ có **nhiều trùng lặp** để thấy bước bỏ trùng `left`/`right` hoạt động:

In [16]:
print("Kết quả:", trace_three_sum([-2, 0, 0, 0, 2, 2, 2]))

Sau sắp xếp: [-2, 0, 0, 0, 2, 2, 2]
i=0(-2) left=1(0) right=6(2) total= 0 -> GHI [-2, 0, 2]
    bỏ trùng left tại 2 (0)
    bỏ trùng left tại 3 (0)
    bỏ trùng right tại 5 (2)
i=1(0) left=2(0) right=6(2) total= 2 -> right -= 1
i=1(0) left=2(0) right=5(2) total= 2 -> right -= 1
i=1(0) left=2(0) right=4(2) total= 2 -> right -= 1
i=1(0) left=2(0) right=3(0) total= 0 -> GHI [0, 0, 0]
i=2: nums[i]=0 trùng phần tử trước -> bỏ qua
i=3: nums[i]=0 trùng phần tử trước -> bỏ qua
i=4: nums[i]=2 > 0 -> dừng hẳn
Kết quả: [[-2, 0, 2], [0, 0, 0]]


# 8. Phiên bản đúng format LeetCode

In [17]:
class Solution:
    def threeSum(self, nums: List[int]) -> List[List[int]]:
        nums.sort()
        n = len(nums)
        result = []
        for i in range(n - 2):
            if nums[i] > 0:
                break
            if i > 0 and nums[i] == nums[i - 1]:
                continue
            left, right = i + 1, n - 1
            while left < right:
                total = nums[i] + nums[left] + nums[right]
                if total < 0:
                    left += 1
                elif total > 0:
                    right -= 1
                else:
                    result.append([nums[i], nums[left], nums[right]])
                    left += 1
                    right -= 1
                    while left < right and nums[left] == nums[left - 1]:
                        left += 1
                    while left < right and nums[right] == nums[right + 1]:
                        right -= 1
        return result

Vì thứ tự kết quả không quan trọng, hàm `normalize` sắp xếp mỗi bộ ba và cả danh sách trước khi so sánh.

In [18]:
def normalize(triples):
    return sorted(sorted(t) for t in triples)

def check(name, actual, expected):
    assert normalize(actual) == normalize(expected), (name, actual, expected)
    assert len(actual) == len({tuple(sorted(t)) for t in actual}), (name, "có bộ trùng", actual)
    print(f"{name:<28} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", sol.threeSum([-1, 0, 1, 2, -1, -4]), [[-1, -1, 2], [-1, 0, 1]])
check("Ví dụ 2", sol.threeSum([0, 1, 1]), [])
check("Ví dụ 3", sol.threeSum([0, 0, 0]), [[0, 0, 0]])
check("Nhiều số 0", sol.threeSum([0, 0, 0, 0, 0]), [[0, 0, 0]])
check("Nhiều trùng", sol.threeSum([-2, 0, 0, 2, 2]), [[-2, 0, 2]])
check("Toàn dương", sol.threeSum([1, 2, 3]), [])
check("Nhiều đáp án", sol.threeSum([-4, -2, -1, 0, 1, 2, 3]),
      [[-4, 1, 3], [-2, -1, 3], [-2, 0, 2], [-1, 0, 1]])

Ví dụ 1                      PASS  [[-1, -1, 2], [-1, 0, 1]]
Ví dụ 2                      PASS  []
Ví dụ 3                      PASS  [[0, 0, 0]]
Nhiều số 0                   PASS  [[0, 0, 0]]
Nhiều trùng                  PASS  [[-2, 0, 2]]
Toàn dương                   PASS  []
Nhiều đáp án                 PASS  [[-4, 1, 3], [-2, -1, 3], [-2, 0, 2], [-1, 0, 1]]


## Đối chiếu với Brute Force

In [19]:
import random

random.seed(15)
for _ in range(2000):
    nums = [random.randint(-6, 6) for _ in range(random.randint(3, 14))]
    assert normalize(sol.threeSum(nums[:])) == normalize(three_sum_bruteforce(nums)), nums
    assert normalize(three_sum_hashset(nums)) == normalize(three_sum_bruteforce(nums)), nums
print("2000 test ngẫu nhiên: 3 cách khớp nhau, không có bộ trùng")

2000 test ngẫu nhiên: 3 cách khớp nhau, không có bộ trùng


# 9. Vì sao thuật toán đúng?

**Không bỏ sót.** Xét một bộ ba hợp lệ bất kỳ, sau khi sắp xếp là `a ≤ b ≤ c` với `a` ở vị trí **đầu tiên** mà giá trị `a` xuất hiện (vị trí `i` không bị bỏ qua).

Trong vòng `while` với `i` đó, hai con trỏ thu hẹp đoạn `[i+1, n-1]`. Lập luận giống bài Container With Most Water / Two Sum II: khi `total < 0`, mọi cặp dùng `nums[left]` với một phần tử ≤ `nums[right]` đều có tổng ≤ hiện tại < 0, nên bỏ `left` an toàn; tương tự với `total > 0`. Vì vậy cặp `(b, c)` sẽ được gặp.

**Không trùng lặp.**

- Mỗi giá trị `a` chỉ được làm số cố định **một lần** (bỏ qua `nums[i] == nums[i-1]`).
- Với `a` cố định, sau khi ghi `(a, b, c)`, ta dời `left` qua mọi giá trị `b` và `right` qua mọi giá trị `c`. Vì `b + c` phải bằng `-a`, khi `b` đã đổi thì `c` cũng phải đổi — bộ ba tiếp theo (nếu có) khác hẳn.

# 10. Độ phức tạp

Gọi `n = len(nums)`.

- Sắp xếp: `O(n log n)`.
- Vòng ngoài `n` lần, mỗi lần hai con trỏ đi tổng cộng `O(n)` bước → `O(n²)`.

```text
Time:  O(n²)
Space: O(1) bộ nhớ phụ ngoài kết quả (không tính bộ nhớ của sort; Python dùng Timsort cần tới O(n))
```

| Cách làm | Thời gian | Bộ nhớ phụ | Bỏ trùng |
|---|---|---|---|
| Brute force | O(n³) | O(số kết quả) | set tuple |
| Cố định + Hash Set | O(n²) | O(n) | set tuple |
| Sắp xếp + Two Pointers | O(n²) | O(1) | so với phần tử kề |

Người ta tin rằng không có thuật toán nhanh hơn `O(n²)` đáng kể cho 3Sum tổng quát — `O(n²)` là mục tiêu của bài.

# 11. Những lỗi dễ mắc

## Lỗi 1: Bỏ trùng số cố định bằng cách so với phần tử **sau**

`if nums[i] == nums[i + 1]: continue` sẽ bỏ lỡ bộ ba dùng **hai** phần tử bằng nhau, ví dụ `[-1, -1, 2]`.

In [20]:
def wrong_skip_next(nums: List[int]) -> List[List[int]]:
    nums = sorted(nums)
    n = len(nums)
    result = []
    for i in range(n - 2):
        if i + 1 < n and nums[i] == nums[i + 1]:   # SAI: so với phần tử SAU
            continue
        left, right = i + 1, n - 1
        while left < right:
            total = nums[i] + nums[left] + nums[right]
            if total < 0:
                left += 1
            elif total > 0:
                right -= 1
            else:
                result.append([nums[i], nums[left], nums[right]])
                left += 1
                right -= 1
                while left < right and nums[left] == nums[left - 1]:
                    left += 1
    return result

print("Sai :", wrong_skip_next([-1, 0, 1, 2, -1, -4]))
print("Đúng:", three_sum([-1, 0, 1, 2, -1, -4]))

Sai : [[-1, 0, 1]]
Đúng: [[-1, -1, 2], [-1, 0, 1]]


## Lỗi 2: Quên bỏ trùng sau khi tìm được bộ ba

In [21]:
def wrong_no_inner_skip(nums: List[int]) -> List[List[int]]:
    nums = sorted(nums)
    n = len(nums)
    result = []
    for i in range(n - 2):
        if i > 0 and nums[i] == nums[i - 1]:
            continue
        left, right = i + 1, n - 1
        while left < right:
            total = nums[i] + nums[left] + nums[right]
            if total < 0:
                left += 1
            elif total > 0:
                right -= 1
            else:
                result.append([nums[i], nums[left], nums[right]])
                left += 1                           # thiếu vòng bỏ trùng
                right -= 1
    return result

print("Sai :", wrong_no_inner_skip([-2, 0, 0, 0, 2, 2, 2]))
print("Đúng:", three_sum([-2, 0, 0, 0, 2, 2, 2]))

Sai : [[-2, 0, 2], [-2, 0, 2], [-2, 0, 2], [0, 0, 0]]
Đúng: [[-2, 0, 2], [0, 0, 0]]


## Lỗi 3: Chỉ dời một con trỏ sau khi tìm được bộ ba

Nếu chỉ `left += 1` mà không `right -= 1`, vẫn đúng (bước sau sẽ tự sửa), nhưng dễ kết hợp sai với vòng bỏ trùng. Quy ước an toàn: **dời cả hai**, rồi bỏ trùng.

## Lỗi 4: Trả về chỉ số thay vì giá trị

Khác Two Sum, 3Sum yêu cầu **giá trị** của các phần tử. (Và sau khi sắp xếp, chỉ số gốc cũng đã mất.)

## Lỗi 5: Dùng `set` của `list`

`{[1, 2, 3]}` báo lỗi `TypeError: unhashable type: 'list'`. Phải đổi sang `tuple`.

# 12. Mẹo nhận diện pattern

**k-Sum = cố định (k - 2) phần tử + Two Pointers:**

- 2Sum trên mảng đã sắp xếp → Two Pointers `O(n)`.
- 3Sum → cố định 1 + 2Sum → `O(n²)`.
- 4Sum (**#18**) → cố định 2 + 2Sum → `O(n³)`.

**Dấu hiệu dùng sắp xếp:** đề cần **giá trị** (không cần chỉ số gốc) và yêu cầu **không trùng lặp**. Sắp xếp đưa giá trị bằng nhau lại gần nhau, bỏ trùng chỉ cần so với phần tử kề.

## Template

```python
nums.sort()
for i in range(n - 2):
    if i > 0 and nums[i] == nums[i - 1]:
        continue
    left, right = i + 1, n - 1
    while left < right:
        total = nums[i] + nums[left] + nums[right]
        if total < target:   left += 1
        elif total > target: right -= 1
        else:
            ghi nhận; left += 1; right -= 1
            while left < right and nums[left] == nums[left - 1]: left += 1
            while left < right and nums[right] == nums[right + 1]: right -= 1
```

Bài liên quan: **#1 Two Sum**, **#167 Two Sum II**, **#16 3Sum Closest**, **#18 4Sum**, **#259 3Sum Smaller**.

# 13. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Lập bảng trace (giống mục 7) cho `nums = [-2, 0, 1, 1, 2]`. Kết quả là gì?

## Bài 2 — Đoán output

```python
print(sorted([3, -1, 0, -1]))
print(tuple(sorted([2, -1, -1])))
print({(1, 2), (1, 2), (2, 1)})
```

## Bài 3 — Điền chỗ trống

In [22]:
def exercise_3_three_sum(nums: List[int]) -> List[List[int]]:
    nums = sorted(nums)
    n = len(nums)
    result = []
    for i in range(n - 2):
        # TODO 1: bỏ qua nếu nums[i] trùng với phần tử TRƯỚC
        left, right = i + 1, n - 1
        while left < right:
            # TODO 2: tính total, dời left / right
            # TODO 3: khi total == 0: ghi nhận, dời cả hai, bỏ trùng left và right
            pass
    return result


# assert normalize(exercise_3_three_sum([-1, 0, 1, 2, -1, -4])) == [[-1, -1, 2], [-1, 0, 1]]
# assert exercise_3_three_sum([0, 0, 0, 0]) == [[0, 0, 0]]

## Bài 4 — Biến thể: 3Sum với `target` bất kỳ

Tổng quát: tìm mọi bộ ba (không trùng) có tổng bằng `target`.

```python
three_sum_target([1, 2, 3, 4, 5], 9) -> [[1, 3, 5], [2, 3, 4]]
three_sum_target([1, 1, 1, 1], 3)    -> [[1, 1, 1]]
```

Lưu ý: tối ưu "dừng khi `nums[i] > 0`" **không còn đúng** khi `target` tùy ý (ví dụ mảng có số âm). Hãy bỏ nó đi.

In [23]:
def exercise_4_three_sum_target(nums: List[int], target: int) -> List[List[int]]:
    # TODO
    pass


# assert normalize(exercise_4_three_sum_target([1, 2, 3, 4, 5], 9)) == [[1, 3, 5], [2, 3, 4]]
# assert exercise_4_three_sum_target([1, 1, 1, 1], 3) == [[1, 1, 1]]
# assert normalize(exercise_4_three_sum_target([-5, -3, -1, 0, 2], -4)) == [[-5, -1, 2], [-3, -1, 0]]

## Bài 5 — Mini challenge: 3Sum Smaller (LeetCode #259)

Đếm số bộ chỉ số `i < j < k` sao cho `nums[i] + nums[j] + nums[k] < target`. Ở bài này **đếm theo chỉ số**, không bỏ trùng giá trị. Yêu cầu `O(n²)`.

```python
three_sum_smaller([-2, 0, 1, 3], 2) -> 2     # [-2,0,1] và [-2,0,3]
three_sum_smaller([], 0)            -> 0
three_sum_smaller([0, 0, 0], 1)     -> 1
```

Gợi ý nhanh: sau khi sắp xếp, nếu `nums[i] + nums[left] + nums[right] < target`, thì **mọi** `k` từ `left + 1` tới `right` cũng thỏa với cặp `(i, left)`.

In [24]:
def exercise_5_three_sum_smaller(nums: List[int], target: int) -> int:
    # TODO
    pass


# assert exercise_5_three_sum_smaller([-2, 0, 1, 3], 2) == 2
# assert exercise_5_three_sum_smaller([], 0) == 0
# assert exercise_5_three_sum_smaller([0, 0, 0], 1) == 1

# 14. Gợi ý

**Bài 1**
- Mức 1: mảng đã sắp xếp sẵn.
- Mức 2: với `i = 0` (giá trị -2), tìm cặp tổng 2 trong `[0, 1, 1, 2]`.
- Mức 3: có hai cặp giá trị khác nhau: `(0, 2)` và `(1, 1)`.

**Bài 3**
- Mức 1: so với code mục 6.
- Mức 2: `if i > 0 and nums[i] == nums[i - 1]: continue`.
- Mức 3: vòng bỏ trùng so `nums[left]` với `nums[left - 1]` (phần tử vừa dùng).

**Bài 4**
- Mức 1: thay mọi `0` bằng `target`.
- Mức 2: xóa dòng `if nums[i] > 0: break`.

**Bài 5**
- Mức 1: cố định `i`, hai con trỏ `left`, `right`.
- Mức 2: khi tổng `< target`: cộng `right - left` vào đáp án, rồi `left += 1`.
- Mức 3: khi tổng `>= target`: `right -= 1`.

# 15. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [25]:
print("Kết quả:", trace_three_sum([-2, 0, 1, 1, 2]))

Sau sắp xếp: [-2, 0, 1, 1, 2]
i=0(-2) left=1(0) right=4(2) total= 0 -> GHI [-2, 0, 2]
i=0(-2) left=2(1) right=3(1) total= 0 -> GHI [-2, 1, 1]
i=1(0) left=2(1) right=4(2) total= 3 -> right -= 1
i=1(0) left=2(1) right=3(1) total= 2 -> right -= 1
i=2: nums[i]=1 > 0 -> dừng hẳn
Kết quả: [[-2, 0, 2], [-2, 1, 1]]


## Đáp án Bài 2

In [26]:
print(sorted([3, -1, 0, -1]))          # [-1, -1, 0, 3]
print(tuple(sorted([2, -1, -1])))      # (-1, -1, 2)
print({(1, 2), (1, 2), (2, 1)})        # 2 phần tử: (1, 2) và (2, 1) là KHÁC nhau với tuple

[-1, -1, 0, 3]
(-1, -1, 2)
{(1, 2), (2, 1)}


Chú ý dòng cuối: tuple **có thứ tự**, `(1, 2) != (2, 1)`. Đó là lý do phải `sorted` trước khi đổi sang `tuple`.

## Đáp án Bài 3

In [27]:
def answer_3_three_sum(nums: List[int]) -> List[List[int]]:
    nums = sorted(nums)
    n = len(nums)
    result = []
    for i in range(n - 2):
        if i > 0 and nums[i] == nums[i - 1]:
            continue
        left, right = i + 1, n - 1
        while left < right:
            total = nums[i] + nums[left] + nums[right]
            if total < 0:
                left += 1
            elif total > 0:
                right -= 1
            else:
                result.append([nums[i], nums[left], nums[right]])
                left += 1
                right -= 1
                while left < right and nums[left] == nums[left - 1]:
                    left += 1
                while left < right and nums[right] == nums[right + 1]:
                    right -= 1
    return result

assert normalize(answer_3_three_sum([-1, 0, 1, 2, -1, -4])) == [[-1, -1, 2], [-1, 0, 1]]
assert answer_3_three_sum([0, 0, 0, 0]) == [[0, 0, 0]]
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [28]:
def answer_4_three_sum_target(nums: List[int], target: int) -> List[List[int]]:
    nums = sorted(nums)
    n = len(nums)
    result = []
    for i in range(n - 2):
        if i > 0 and nums[i] == nums[i - 1]:
            continue
        left, right = i + 1, n - 1
        while left < right:
            total = nums[i] + nums[left] + nums[right]
            if total < target:
                left += 1
            elif total > target:
                right -= 1
            else:
                result.append([nums[i], nums[left], nums[right]])
                left += 1
                right -= 1
                while left < right and nums[left] == nums[left - 1]:
                    left += 1
                while left < right and nums[right] == nums[right + 1]:
                    right -= 1
    return result

assert normalize(answer_4_three_sum_target([1, 2, 3, 4, 5], 9)) == [[1, 3, 5], [2, 3, 4]]
assert answer_4_three_sum_target([1, 1, 1, 1], 3) == [[1, 1, 1]]
assert normalize(answer_4_three_sum_target([-5, -3, -1, 0, 2], -4)) == [[-5, -1, 2], [-3, -1, 0]]
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [29]:
def answer_5_three_sum_smaller(nums: List[int], target: int) -> int:
    nums = sorted(nums)
    n = len(nums)
    count = 0
    for i in range(n - 2):
        left, right = i + 1, n - 1
        while left < right:
            if nums[i] + nums[left] + nums[right] < target:
                count += right - left       # mọi k trong (left, right] đều thỏa
                left += 1
            else:
                right -= 1
    return count

assert answer_5_three_sum_smaller([-2, 0, 1, 3], 2) == 2
assert answer_5_three_sum_smaller([], 0) == 0
assert answer_5_three_sum_smaller([0, 0, 0], 1) == 1

from itertools import combinations
for _ in range(1000):
    nums = [random.randint(-5, 5) for _ in range(random.randint(0, 9))]
    t = random.randint(-6, 6)
    assert answer_5_three_sum_smaller(nums, t) == sum(1 for c in combinations(nums, 3) if sum(c) < t)
print("Bài 5 PASS (kể cả 1000 test so với itertools.combinations)")

Bài 5 PASS (kể cả 1000 test so với itertools.combinations)


# 16. Tổng kết

> **Sắp xếp** → **cố định một số** → **Two Pointers** cho hai số còn lại → **bỏ trùng bằng cách so với phần tử kề**.

---

## Checklist trước khi nộp

- [ ] Đã sắp xếp mảng?
- [ ] Bỏ trùng số cố định bằng `nums[i] == nums[i - 1]` (phần tử **trước**)?
- [ ] Sau khi ghi nhận, dời **cả hai** con trỏ rồi bỏ trùng?
- [ ] Trả về **giá trị**, không phải chỉ số?
- [ ] Đã thử `[0, 0, 0]`, `[0, 0, 0, 0]`, mảng toàn dương, mảng nhiều trùng?